# 01 - Data Cleaning: Motorcycle Insurance
Load the raw data, rename columns to clear English names, check data quality, and save a cleaned version for the next notebooks.

In [7]:
from pathlib import Path

import pandas as pd

RAW_PATH = Path("../data/raw/Motorcycle Insurance.xlsx")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_excel(RAW_PATH)
print(df_raw.shape)
df_raw.head()

(64548, 10)


,rownames,agarald,kon,zon,mcklass,fordald,bonuskl,duration,antskad,skadkost
0,1,0,M,1,4,12,1,0.175342,0,0
1,2,4,M,3,6,9,1,0.000000,0,0
2,3,5,K,3,3,18,1,0.454795,0,0
3,4,5,K,4,1,25,1,0.172603,0,0
4,5,6,K,2,1,26,1,0.180822,0,0


## 1. Look at the original columns

The raw file uses short Swedish variable codes. Check the shape, types, and summary stats before touching anything.

In [8]:
print("Columns:", list(df_raw.columns))
print()
print(df_raw.dtypes)
print()
df_raw.describe(include="all").T

Columns: ['rownames', 'agarald', 'kon', 'zon', 'mcklass', 'fordald', 'bonuskl', 'duration', 'antskad', 'skadkost']

rownames      int64
agarald       int64
kon             str
zon           int64
mcklass       int64
fordald       int64
bonuskl       int64
duration    float64
antskad       int64
skadkost      int64
dtype: object



,count,unique,top,freq,mean,std,min,25%,50%,75%,max
rownames,64548.0,NaN,NaN,NaN,32274.5,18633.546925,1.0,16137.75,32274.5,48411.25,64548.0
agarald,64548.0,NaN,NaN,NaN,42.416062,12.98096,0.0,31.0,44.0,52.0,92.0
kon,64548,2,M,54695,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zon,64548.0,NaN,NaN,NaN,3.213113,1.354591,1.0,2.0,3.0,4.0,7.0
mcklass,64548.0,NaN,NaN,NaN,3.700053,1.523921,1.0,3.0,4.0,5.0,7.0
fordald,64548.0,NaN,NaN,NaN,12.540063,9.727445,0.0,5.0,12.0,16.0,99.0
bonuskl,64548.0,NaN,NaN,NaN,4.024571,2.364742,1.0,2.0,4.0,7.0,7.0
duration,64548.0,NaN,NaN,NaN,1.010671,1.307424,0.0,0.463014,0.827397,1.0,31.33973
antskad,64548.0,NaN,NaN,NaN,0.010798,0.107323,0.0,0.0,0.0,0.0,2.0
skadkost,64548.0,NaN,NaN,NaN,264.017785,4694.693604,0.0,0.0,0.0,0.0,365347.0


## 2. Rename columns to clear English names

The original column names are short Swedish codes (`agarald`, `kon`, `skadkost`, ...). Renaming them makes the analysis readable for anyone, without needing the data dictionary open next to it.

In [17]:
rename_map = {
    "rownames": "policy_id",
    "agarald": "owner_age",
    "kon": "gender",
    "zon": "zone",
    "mcklass": "vehicle_class",
    "fordald": "vehicle_age",
    "bonuskl": "bonus_class",
    "duration": "duration_years",
    "antskad": "claim_count",
    "skadkost": "claim_cost",
}

df = df_raw.rename(columns=rename_map)

# M = "Man", K = "Kvinna" (Swedish for woman) -> spell them out for readability
df["gender"] = df["gender"].map({"M": "Male", "K": "Female"})

df.head()


,policy_id,owner_age,gender,zone,vehicle_class,vehicle_age,bonus_class,duration_years,claim_count,claim_cost
0,1,0,Male,1,4,12,1,0.175342,0,0
1,2,4,Male,3,6,9,1,0.000000,0,0
2,3,5,Female,3,3,18,1,0.454795,0,0
3,4,5,Female,4,1,25,1,0.172603,0,0
4,5,6,Female,2,1,26,1,0.180822,0,0


## 3. Data quality checks

Before trusting this data, check for missing values, duplicates, and any values outside the expected ranges from the data description.

In [10]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Duplicate rows (ignoring policy_id):", df.drop(columns=["policy_id"]).duplicated().sum())
print()
print("gender counts:\n", df["gender"].value_counts())
print()
for col in ["zone", "vehicle_class", "bonus_class"]:
    print(f"{col} unique values:", sorted(df[col].unique()))
print()
print("owner_age range:", df["owner_age"].min(), "-", df["owner_age"].max())
print("vehicle_age range:", df["vehicle_age"].min(), "-", df["vehicle_age"].max())
print()
print("claim_count value counts:\n", df["claim_count"].value_counts())

Missing values per column:
policy_id         0
owner_age         0
gender            0
zone              0
vehicle_class     0
vehicle_age       0
bonus_class       0
duration_years    0
claim_count       0
claim_cost        0
dtype: int64

Duplicate rows (ignoring policy_id): 0

gender counts:
 gender
Male      54695
Female     9853
Name: count, dtype: int64

zone unique values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
vehicle_class unique values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
bonus_class unique values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]

owner_age range: 0 - 92
vehicle_age range: 0 - 99

claim_count value counts:
 claim_count
0    63878
1      643
2       27
Name: count, dtype: int64


## 4. Known data issues to flag

- **`duration_years == 0`** for 2,074 policies (~3.2%) — zero-exposure policies (e.g. a policy that started and was cancelled the same day). They can't be used in a frequency/severity model since exposure can't be zero in the denominator.
- Of those zero-exposure rows, **4 still have a claim recorded** (`claim_count` = 1). This is a genuine data quirk (most likely a very short but non-zero exposure that rounded down to `0.000000`) — it is kept and flagged with a column rather than deleted, so nothing is silently thrown away.
- **`owner_age == 0`** for exactly 1 policy — a single outlier, not worth dropping, just noted here.
- No missing values, no duplicate rows, and `zone` / `vehicle_class` / `bonus_class` are exactly the 1-7 categories from the data description. The rest of the data is clean.

In [11]:
df["is_zero_exposure"] = df["duration_years"] == 0

print("Zero-exposure policies:", df["is_zero_exposure"].sum())
print(
    "Zero-exposure policies that still have a claim recorded:",
    ((df["is_zero_exposure"]) & (df["claim_count"] > 0)).sum(),
)

Zero-exposure policies: 2074
Zero-exposure policies that still have a claim recorded: 4


## 5. Save the cleaned data

Save the renamed, flagged dataset to `data/processed/` so the next notebook (EDA) can load it directly without repeating this cleaning step.

In [12]:
output_path = PROCESSED_DIR / "motorcycle_clean.csv"
df.to_csv(output_path, index=False)

print(f"Saved cleaned data to: {output_path.resolve()}")
print(f"Final shape: {df.shape}")
df.head()

Saved cleaned data to: C:\Users\karaketk\Desktop\Practice Project\Motorcycle Insurance\data\processed\motorcycle_clean.csv
Final shape: (64548, 11)


,policy_id,owner_age,gender,zone,vehicle_class,vehicle_age,bonus_class,duration_years,claim_count,claim_cost,is_zero_exposure
0,1,0,Male,1,4,12,1,0.175342,0,0,False
1,2,4,Male,3,6,9,1,0.000000,0,0,True
2,3,5,Female,3,3,18,1,0.454795,0,0,False
3,4,5,Female,4,1,25,1,0.172603,0,0,False
4,5,6,Female,2,1,26,1,0.180822,0,0,False


## Summary

- Renamed all columns from Swedish codes to clear English names (e.g. `agarald` -> `owner_age`, `skadkost` -> `claim_cost`).
- Mapped `gender` codes to `Male` / `Female`.
- Verified: no missing values, no duplicate rows, all categorical columns match the expected 1-7 ranges from the data description.
- Flagged 2,074 zero-exposure policies with a new `is_zero_exposure` column (kept, not deleted) — these should be excluded from the frequency/severity models later.
- Saved the cleaned dataset to `data/processed/motorcycle_clean.csv`, ready for the next notebook: exploratory data analysis (EDA).